## LEVEL 2
1. Predict google place type from business name (if primaryType is non-suggestive)
2. Assign cuisineType for each place where possible - rest unspecified
3. Assign venueType for each place
4. Export places by resolved and unresolved

#### Initialize

In [3]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
LEVEL1_BUCKET_PATH = PARENT / "server/out/places_level1"
LEVEL2_BUCKET_PATH = PARENT / "server/out/places_level2"
LEVEL2_BUCKET_PATH.mkdir(parents=True, exist_ok=True)
LEVEL1_BUCKET = [f for f in LEVEL1_BUCKET_PATH.rglob("*.csv") if f.is_file()]
DF_LEVEL1 = pd.concat([pd.read_csv(f) for f in LEVEL1_BUCKET], ignore_index=True)
DF_LEVEL1.drop_duplicates(subset=["id"], inplace=True)

#### Parse CHAINS

In [4]:
from server.scripts.clean_places_level_2.find_chain_by_register.find_chain_by_register import find_chain_by_register
from server.scripts.clean_places_level_2.find_chain_by_pattern.find_chain_by_pattern import find_chain_by_pattern

df_parsed_chain = DF_LEVEL1.copy()
# FIND BLOCK CHAINS FROM KNOWN LIST
df_parsed_chain = find_chain_by_register(df_parsed_chain)

# FIND BLOCK CHAINS FROM DUPLIICATE NAME
# AGGREGATE all the rows with first word of display name matching
df_nonchain = df_parsed_chain[df_parsed_chain['is_chain'] == False]
matched = find_chain_by_pattern(df_nonchain, threshold=85, min_matches=2)
matched.to_csv(LEVEL2_BUCKET_PATH / "matched.csv", index=False)
for row in matched.itertuples(index=False):
    df_parsed_chain.loc[row.idx, 'is_chain'] = True
    df_parsed_chain.loc[row.idx, 'chain_name'] = row.name
    df_parsed_chain.loc[row.idx, 'chain_count'] = row.count

#### Parse TYPES

In [5]:
from server.scripts.clean_places_level_2.predict_cuisine_from_name.predict_cuisine_from_name import predict_cuisine_from_name
from server.scripts.clean_places_level_2.map_cuisine.type_parsing import parse_type, check_takeaway

df_parsed_type = df_parsed_chain.copy()

# PREDICT CUISINE TYPE FROM NAME 
mask = df_parsed_type["predictedType"].fillna("").eq("")
df_parsed_type.loc[mask, "predictedType"] = df_parsed_type.loc[mask].apply(predict_cuisine_from_name, axis=1)
df_parsed_type["cuisineType"] = df_parsed_type.apply(parse_type, axis=1)
df_parsed_type["venueType"] = df_parsed_type.apply(check_takeaway, axis=1)

# ── Diagnostics ───────────────────────────────────────────────────────────────
dist = df_parsed_type["cuisineType"].value_counts()
unresolved = (df_parsed_type["cuisineType"] == "Unspecified").sum()
chains = df_parsed_type["is_chain"].sum()
print(f"Chains detected      : {chains} / {len(df_parsed_type)}  ({chains/len(df_parsed_type):.1%})")
print(f"Unique cuisineTypes  : {dist.nunique()}")
print(f"Still 'Unspecified'  : {unresolved} / {len(df_parsed_type)}  ({unresolved/len(df_parsed_type):.1%})")

Chains detected      : 4350 / 18238  (23.9%)
Unique cuisineTypes  : 41
Still 'Unspecified'  : 2377 / 18238  (13.0%)


In [6]:
# # ── Export Unspecified ─────────────────────────────────────────────────────
# df_unspecified = df_parsed_type[df_parsed_type["cuisineType"] == "Unspecified"]
# df_unspecified[["displayName", "chain_name", "predictedType", "venueType", 
#     "googleMapsUri", "websiteUri"]].to_csv("unspecified.csv", index=False)

#### Export

In [7]:
df_level2 = df_parsed_type.copy()
df_resolved = df_level2[df_level2["cuisineType"]!="Unspecified"]
df_resolved.to_csv(LEVEL2_BUCKET_PATH / "places_resolved.csv", index=False)
df_unresolved = df_level2[df_level2["cuisineType"]=="Unspecified"]
df_unresolved.to_csv(LEVEL2_BUCKET_PATH / "places_unresolved.csv", index=False)